# FedKG-Maint — Phase 2: Centralized baselines and evaluation toolkit (prep_v3)

**Input:** verified Phase 1 data in `prep_v3/`. This phase covers centralized baselines only; it does not implement federated learning, privacy, the knowledge graph, or the LLM.

## Questions answered
1. Do simple prevalence and recency baselines provide useful signal?
2. How well do centralized Logistic Regression and HistGradientBoosting perform as references for later federated models?
3. What is the contribution of maintenance-recency features (`full` vs `no_maintenance`)?
4. How do the `global` and `local` normalization policies differ?
5. How uncertain are the estimates when machines—not hourly rows—are the bootstrap unit?

## Evaluation protocol

- **Selection set:** `validation` only. Hyperparameters, heuristic direction, and alert thresholds are chosen without using held-out test labels.
- **Primary held-out-site result:** `test_site_full` (factory 5 after the separate unlabeled calibration period). In prep_v3, this set is disjoint from the local-scaler calibration artifact, so both global and local policies may be reported.
- **Strict post-training-time held-out-site result:** `test_site_post`, a subset of `test_site_full` at or after the end of the supervised training window.
- **Temporal result on development factories:** `test_time_late`.
- **Late held-out-site sensitivity result:** `test_site_late`, a subset of `test_site_full` with few events.

`test_site_full`, `test_site_post`, and `test_site_late` are nested, not independent datasets. Do not combine them as independent evidence. Confidence intervals use a cluster bootstrap over machines. PR-AUC (average precision) is primary; lift over prevalence, ROC-AUC, thresholded precision/recall/F1/FPR, and event recall are secondary. The constant prevalence baseline has no operational threshold, so thresholded metrics are reported as N/A for it.

The notebook writes to `phase2_baselines_prep_v3/` to preserve earlier Phase 2 artifacts.

In [ ]:
# 1.1 Setup, single CONFIG, directories, checks
import os, sys, json, time, importlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

try:  # Google Colab
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:  # local run / testing
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

CONFIG = {
    "SEED": 42,
    "BOOTSTRAP_B": 1000,                       # machine-level bootstrap replicates
    "CI_LEVEL": 0.95,
    "EVAL_SETS": ["validation", "test_site_full", "test_site_post", "test_time_late", "test_site_late"],
    "POLICIES": ["global", "local"],
    "FEATURE_SETS": ["full", "no_maintenance"],
    "LR_GRID_C": [0.01, 0.1, 1.0],
    "HGB_GRID": [
        dict(learning_rate=0.1, max_depth=3, max_iter=150),
        dict(learning_rate=0.1, max_depth=6, max_iter=150),
        dict(learning_rate=0.05, max_depth=3, max_iter=300),
        dict(learning_rate=0.05, max_depth=6, max_iter=300),
    ],
    "FRAGILE_EVENTS": 30,
}
if os.environ.get("FEDKG_FAST"):              # smoke test only; not for final results
    CONFIG.update(BOOTSTRAP_B=60, LR_GRID_C=[0.1], HGB_GRID=[dict(learning_rate=0.1, max_depth=3, max_iter=40)])

PREP_DIR = PROJECT_DIR / "prep_v3"
RAW_DIR = PROJECT_DIR / "data" / "raw"
P2_DIR = PROJECT_DIR / "phase2_baselines_prep_v3"
for d in ["results", "predictions", "models", "reports"]:
    (P2_DIR / d).mkdir(parents=True, exist_ok=True)

CHECKS = {}
def record(check_id, ok, detail=""):
    CHECKS[check_id] = {"status": "PASS" if ok else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {check_id}: {detail}")
    if not ok:
        raise AssertionError(f"Check failed: {check_id} — {detail}")

required = [
    PREP_DIR / "processed" / name
    for name in ["train.parquet", "validation.parquet", "test_time_late.parquet", "test_site_late.parquet",
                 "test_site_full.parquet", "heldout_calibration_features.parquet"]
]
required += [PREP_DIR / "configs" / "feature_groups.json", PREP_DIR / "fedkg_data.py"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run the revised Phase-1 notebook first. Missing prep_v3 artifacts:\n" + "\n".join(missing))

sys.path.insert(0, str(PREP_DIR))
sys.modules.pop("fedkg_data", None)  # prevent a stale prep_v2 module in a reused Colab kernel
importlib.invalidate_caches()
import fedkg_data
record("P3-0 loader imported from prep_v3", Path(fedkg_data.__file__).resolve().parent == PREP_DIR.resolve(), fedkg_data.__file__)
FEATURE_CFG = json.loads((PREP_DIR / "configs" / "feature_groups.json").read_text())
ALL_FEATURES = FEATURE_CFG["feature_sets"]["full"]
TARGET = FEATURE_CFG["target"]
TRAIN_CUTOFF = pd.Timestamp(FEATURE_CFG["boundaries"]["train_cutoff"])
np.random.seed(CONFIG["SEED"])

failures = pd.read_csv(RAW_DIR / "PdM_failures.csv", parse_dates=["datetime"])
failures["datetime"] = failures["datetime"].astype("datetime64[ns]")
fail_events = (failures[["machineID", "datetime"]].drop_duplicates()
               .sort_values(["machineID", "datetime"]).reset_index(drop=True))

# Enforce the revised prep_v3 calibration contract before loading model features.
calibration = pd.read_parquet(PREP_DIR / "processed" / "heldout_calibration_features.parquet")
site_full_raw = pd.read_parquet(PREP_DIR / "processed" / "test_site_full.parquet")
key_cols = ["machineID", "datetime"]
cal_overlap = calibration[key_cols].merge(site_full_raw[key_cols], on=key_cols, how="inner")
record("P3-1 held-out calibration is unlabeled and disjoint from test_site_full",
       TARGET not in calibration.columns and cal_overlap.empty and calibration["datetime"].max() < site_full_raw["datetime"].min(),
       f"calibration rows={len(calibration):,}; overlap rows={len(cal_overlap)}; first test row={site_full_raw['datetime'].min()}")
record("P3-2 v3 feature configuration is complete",
       len(ALL_FEATURES) == 25 and set(FEATURE_CFG["feature_sets"]["no_maintenance"]).issubset(ALL_FEATURES)
       and set(FEATURE_CFG["feature_sets"]["no_maintenance"]) != set(ALL_FEATURES),
       f"features full={len(ALL_FEATURES)}; no_maintenance={len(FEATURE_CFG['feature_sets']['no_maintenance'])}")

print("pandas", pd.__version__, "| numpy", np.__version__, "| project:", PROJECT_DIR)
print("Input preparation:", PREP_DIR)
print("Output directory:", P2_DIR)
print("Distinct raw failure events:", len(fail_events))
print("Training cutoff:", TRAIN_CUTOFF)

## 2. Evaluation toolkit (written to `phase2_baselines_prep_v3/fedkg_eval.py`, reused by every later phase)

Metrics are computed from **machine weights**: a bootstrap replicate is just an integer weight per machine (how many times it was drawn), so thousands of replicates are cheap and every comparison between two models can use the identical resamples. The implementation handles tied scores exactly like scikit-learn and is verified against it (and against physically replicated rows) before any model is trained.

In [ ]:
# 2.1  Write the evaluation module, import it, and verify it against scikit-learn
MODULE_SRC = '''"""Evaluation toolkit for FedKG-Maint (machine-level weighted metrics + cluster bootstrap).

    import sys; sys.path.insert(0, "<PROJECT_DIR>/phase2_baselines_prep_v3")
    from fedkg_eval import SetEval, draw_weights
    se = SetEval(meta, y, fail_events)          # meta: DataFrame with machineID, datetime sorted by (machine, time)
    pk = se.pack(scores); thr = se.best_f1_threshold(pk)       # threshold: fit on VALIDATION only
    point = se.metrics(pk, None, thr)
    reps = se.bootstrap(pk, thr, draw_weights(se.M, 1000, seed))
"""
import numpy as np
import pandas as pd

METRIC_COLS = ["ap", "roc_auc", "precision", "recall", "f1", "fpr", "event_recall"]


def draw_weights(n_machines, B, seed):
    """B bootstrap resamples of machines: integer counts per machine (rows sum to n_machines)."""
    rng = np.random.default_rng(seed)
    return rng.multinomial(n_machines, np.full(n_machines, 1.0 / n_machines), size=B).astype("float64")


def ci(values, level=0.95):
    a = (1.0 - level) / 2.0
    return tuple(np.percentile(np.asarray(values, dtype="float64"), [100 * a, 100 * (1 - a)]))


class SetEval:
    def __init__(self, meta, y, fail_events, horizon_hours=24):
        self.y = np.asarray(y, dtype="float64")
        self.n = len(self.y)
        mid = meta["machineID"].to_numpy()
        self.machines, self.mach_idx = np.unique(mid, return_inverse=True)
        self.mach_idx = self.mach_idx.ravel()
        self.M = len(self.machines)
        ts = meta["datetime"].to_numpy()
        assert np.all(np.diff(self.mach_idx) >= 0), "rows must be sorted by machine"
        assert np.all((np.diff(ts.astype("int64")) >= 0) | (np.diff(self.mach_idx) > 0)), "rows must be sorted by time inside a machine"
        starts = np.searchsorted(self.mach_idx, np.arange(self.M), side="left")
        ends = np.searchsorted(self.mach_idx, np.arange(self.M), side="right")
        H = np.timedelta64(int(horizon_hours), "h")
        slices, ev_m = [], []
        fe = fail_events[fail_events["machineID"].isin(self.machines)]
        for m, g in fe.groupby("machineID"):
            k = int(np.searchsorted(self.machines, m))
            s, e = int(starts[k]), int(ends[k])
            t_m = ts[s:e]
            f = g["datetime"].to_numpy()
            lo = np.searchsorted(t_m, f - H, side="left")      # rows with t >= f - 24h
            hi = np.searchsorted(t_m, f, side="left")          # rows with t <  f
            for a, b in zip(lo, hi):
                if b > a:
                    slices.append((s + int(a), s + int(b)))
                    ev_m.append(k)
        self.ev_slices = slices
        self.ev_mach = np.array(ev_m, dtype=int)
        self.n_events = len(slices)

    def pack(self, score):
        score = np.asarray(score, dtype="float64")
        if not np.isfinite(score).all():
            raise ValueError("non-finite scores")
        uniq, inv = np.unique(-score, return_inverse=True)     # group 0 = highest score; ties share a group
        return {"score": score, "inv": np.asarray(inv).ravel().astype(np.int64), "G": len(uniq), "thr_vals": -uniq}

    def best_f1_threshold(self, pk):
        tp_g = np.bincount(pk["inv"], self.y, pk["G"])
        fp_g = np.bincount(pk["inv"], 1.0 - self.y, pk["G"])
        tp, fp = np.cumsum(tp_g), np.cumsum(fp_g)
        f1 = 2.0 * tp / (tp + fp + tp[-1])
        return float(pk["thr_vals"][int(np.argmax(f1))])

    def _event_flags(self, pk, thr):
        if self.n_events == 0:
            return np.zeros(0, dtype=bool)
        if "ev_max" not in pk:
            pk["ev_max"] = np.array([pk["score"][a:b].max() for a, b in self.ev_slices])
        return pk["ev_max"] >= thr

    def metrics(self, pk, wm=None, thr=None):
        wr = np.ones(self.n) if wm is None else wm[self.mach_idx]
        wy = wr * self.y
        wn = wr - wy
        tp_g = np.bincount(pk["inv"], wy, pk["G"])
        fp_g = np.bincount(pk["inv"], wn, pk["G"])
        tp, fp = np.cumsum(tp_g), np.cumsum(fp_g)
        P, N = float(tp[-1]), float(fp[-1])
        out = {"prevalence": P / (P + N) if P + N > 0 else np.nan}
        if P <= 0 or N <= 0:
            out.update({k: np.nan for k in METRIC_COLS})
            return out
        denom = tp + fp
        prec = np.divide(tp, denom, out=np.zeros_like(tp), where=denom > 0)
        rec = tp / P
        out["ap"] = float(np.sum(np.diff(np.concatenate(([0.0], rec))) * prec))
        out["roc_auc"] = float(np.sum(tp_g * (N - fp + 0.5 * fp_g)) / (P * N))
        if thr is not None:
            pred = pk["score"] >= thr
            tpt, fpt = float(np.sum(wy * pred)), float(np.sum(wn * pred))
            p_ = tpt / (tpt + fpt) if tpt + fpt > 0 else 0.0
            r_ = tpt / P
            out.update({"precision": p_, "recall": r_, "f1": 2 * p_ * r_ / (p_ + r_) if p_ + r_ > 0 else 0.0, "fpr": fpt / N})
            wev = np.ones(self.n_events) if wm is None else wm[self.ev_mach]
            flag = self._event_flags(pk, thr)
            out["event_recall"] = float(np.sum(wev * flag) / np.sum(wev)) if np.sum(wev) > 0 else np.nan
        else:
            # A constant prevalence score has no meaningful operating threshold.
            out.update({k: np.nan for k in ["precision", "recall", "f1", "fpr", "event_recall"]})
        return out

    def bootstrap(self, pk, thr, W):
        return pd.DataFrame([self.metrics(pk, w, thr) for w in W])
'''
(P2_DIR / "fedkg_eval.py").write_text(MODULE_SRC)
sys.path.insert(0, str(P2_DIR))
sys.modules.pop("fedkg_eval", None)  # ensure the module just written to this versioned output directory is used
importlib.invalidate_caches()
import fedkg_eval
from fedkg_eval import SetEval, draw_weights, ci

# ---- verification against scikit-learn (with tied scores) and against physically replicated rows
rng = np.random.default_rng(CONFIG["SEED"])
n_m, per = 20, 200
meta_t = pd.DataFrame({"machineID": np.repeat(np.arange(1, n_m + 1), per),
                       "datetime": np.tile(pd.date_range("2015-01-01", periods=per, freq="h").to_numpy(), n_m)})
y_t = (rng.random(n_m * per) < 0.08).astype(float)
s_t = np.round(1.5 * y_t + rng.normal(0, 1, n_m * per), 1)                       # many ties on purpose
no_events = pd.DataFrame({"machineID": pd.Series(dtype="int64"), "datetime": pd.Series(dtype="datetime64[ns]")})
se_t = SetEval(meta_t, y_t, no_events)
m0 = se_t.metrics(se_t.pack(s_t))
ok_sk = abs(m0["ap"] - average_precision_score(y_t, s_t)) < 1e-9 and abs(m0["roc_auc"] - roc_auc_score(y_t, s_t)) < 1e-9
counts = rng.multinomial(n_m, np.full(n_m, 1 / n_m))
rows_rep = np.repeat(np.arange(len(y_t)), counts[se_t.mach_idx].astype(int))
mw = se_t.metrics(se_t.pack(s_t), counts.astype(float))
ok_rep = (abs(mw["ap"] - average_precision_score(y_t[rows_rep], s_t[rows_rep])) < 1e-9
          and abs(mw["roc_auc"] - roc_auc_score(y_t[rows_rep], s_t[rows_rep])) < 1e-9)
record("E-1 weighted AP / ROC-AUC equal scikit-learn (ties) and equal metrics on physically replicated rows", bool(ok_sk and ok_rep),
       f"AP={m0['ap']:.6f} vs sklearn {average_precision_score(y_t, s_t):.6f}")

# event window: failure at hour 50 -> rows t with t < f <= t + 24h  =>  rows 26..49
meta_e = meta_t[meta_t["machineID"] == 1].reset_index(drop=True)
fe_t = pd.DataFrame({"machineID": [1], "datetime": [meta_e["datetime"].iloc[50]]})
se_e = SetEval(meta_e, np.zeros(len(meta_e)), fe_t)
record("E-2 event window = the 24 rows strictly before the failure", se_e.ev_slices == [(26, 50)], f"slice={se_e.ev_slices}")

zero_w = se_t.metrics(se_t.pack(s_t), np.r_[np.zeros(n_m - 1), n_m].astype(float))   # one machine only: must not crash
print("Toolkit written to", P2_DIR / "fedkg_eval.py")

## 3. Load the `prep_v3` splits and define evaluation sets

`test_site_post` is derived from `test_site_full` by restricting rows to timestamps at or after the end of the supervised training window. This is a nested subset, not an independent test. Unlike prep_v2, `test_site_full` in prep_v3 already begins after the held-out calibration window, so the local-scaler evaluation is disjoint from calibration and can be reported for the full site test.

In [ ]:
# 3.1 Load all policy/feature-set combinations; verify labels and metadata align
BASE_SPLITS = ["validation", "test_time_late", "test_site_late", "test_site_full"]
DATA = {}                                          # (policy, feature_set) -> {split: array(s)}
META_BASE, Y_BASE = {}, {}

for policy in CONFIG["POLICIES"]:
    for fs in CONFIG["FEATURE_SETS"]:
        d = {}
        Xtr, ytr, mtr = fedkg_data.load_split("train", policy=policy, feature_set=fs)
        d["train"] = (Xtr, ytr)
        for split in BASE_SPLITS:
            X, y, meta = fedkg_data.load_split(split, policy=policy, feature_set=fs)
            d[split] = X
            if split not in META_BASE:
                META_BASE[split], Y_BASE[split] = meta.copy(), y.copy()
            else:
                assert meta.equals(META_BASE[split]), f"metadata order mismatch for {split}/{policy}/{fs}"
                assert np.array_equal(y, Y_BASE[split]), f"labels differ across loader policies/features for {split}"
        DATA[(policy, fs)] = d

post_mask = (META_BASE["test_site_full"]["datetime"] >= TRAIN_CUTOFF).to_numpy()
assert post_mask.any(), "test_site_post is empty; verify prep_v3 boundaries"
for combo in DATA:
    DATA[combo]["test_site_post"] = DATA[combo]["test_site_full"][post_mask]

EVAL = {}
for split in CONFIG["EVAL_SETS"]:
    if split == "test_site_post":
        meta = META_BASE["test_site_full"].loc[post_mask].reset_index(drop=True)
        y = Y_BASE["test_site_full"][post_mask]
    else:
        meta = META_BASE[split].reset_index(drop=True)
        y = Y_BASE[split]
    EVAL[split] = {"meta": meta, "y": np.asarray(y), "se": SetEval(meta, y, fail_events)}

rows = []
for split, item in EVAL.items():
    rows.append({"eval_set": split, "rows": len(item["y"]), "machines": item["se"].M,
                 "positive_rows": int(item["y"].sum()), "prevalence_%": round(100 * item["y"].mean(), 3),
                 "failure_events": item["se"].n_events,
                 "fragile": item["se"].n_events < CONFIG["FRAGILE_EVENTS"]})
EVAL_TABLE = pd.DataFrame(rows)
print(EVAL_TABLE.to_string(index=False))

record("D-1 held-out sets contain only factory_5; validation/time-test exclude it",
       all(set(EVAL[s]["meta"]["client_id"]) == {"factory_5"} for s in ["test_site_full", "test_site_post", "test_site_late"])
       and all("factory_5" not in set(EVAL[s]["meta"]["client_id"]) for s in ["validation", "test_time_late"]))
record("D-2 test_site_post is a subset of test_site_full and starts after train cutoff",
       EVAL["test_site_post"]["meta"]["datetime"].min() >= TRAIN_CUTOFF
       and len(EVAL["test_site_post"]["meta"]) < len(EVAL["test_site_full"]["meta"]),
       f"site_full={len(EVAL['test_site_full']['y']):,}; site_post={len(EVAL['test_site_post']['y']):,}; cutoff={TRAIN_CUTOFF}")
record("D-3 test_site_late is nested within test_site_full (not an independent sample)",
       set(map(tuple, EVAL["test_site_late"]["meta"][key_cols].to_numpy())).issubset(
           set(map(tuple, EVAL["test_site_full"]["meta"][key_cols].to_numpy()))))

## 4. Reference baselines (no learned model)

The constant prevalence baseline is a ranking baseline, not an actionable alerting policy; its thresholded metrics are deliberately N/A. The error-recency and maintenance-recency baselines each choose their score direction using **validation only**, then lock that direction before any held-out test metrics are computed.

In [ ]:
# 4.1 Define the pre-specified baselines; choose heuristic directions using validation only
SCORES, INFO, SCORING_LOG, SEL_ROWS = {}, {}, [], []
BASELINE_DEFS = {}

def register(key, model, fs, policy, params, score_by_set):
    SCORES[key] = score_by_set
    INFO[key] = {"model": model, "feature_set": fs, "policy": policy, "params": params}

# Unscaled features are used only for interpretable recency heuristics.
RAW = {}
for split in BASE_SPLITS:
    RAW[split] = fedkg_data.load_split(split, policy="none", feature_set="full")[0]
RAW["test_site_post"] = RAW["test_site_full"][post_mask]
col = {name: i for i, name in enumerate(ALL_FEATURES)}

BASELINE_DEFS["prevalence|full|none"] = {"model": "prevalence", "feature": None, "sign": 0.0, "direction": "constant"}
for short, feature in [
    ("heur_error_recency", "hours_since_last_error_log1p"),
    ("heur_maint_recency", "hours_since_last_maintenance_log1p"),
]:
    val_x = RAW["validation"][:, col[feature]].astype("float64")
    val_y = EVAL["validation"]["y"]
    auc_recent_risky = float(roc_auc_score(val_y, -val_x))
    sign = -1.0 if auc_recent_risky >= 0.5 else 1.0
    direction = "shorter time => riskier" if sign < 0 else "longer time => riskier"
    key = f"{short}|full|none"
    BASELINE_DEFS[key] = {"model": short, "feature": feature, "sign": sign,
                          "direction": direction, "validation_auc_after_orientation": max(auc_recent_risky, 1 - auc_recent_risky)}

for key, p in BASELINE_DEFS.items():
    INFO[key] = {"model": p["model"], "feature_set": "full", "policy": "none", "params": p.copy()}

print("Locked validation-only heuristic directions:")
for key, p in BASELINE_DEFS.items():
    if p["model"] != "prevalence":
        print(f"  {key}: {p['direction']}; validation ROC-AUC after orientation={p['validation_auc_after_orientation']:.3f}")
print("Prevalence baseline: constant score, no alert threshold.")

## 5. Centralized models: Logistic Regression and HistGradientBoosting

For every predeclared combination of model family, normalization policy, and feature set, hyperparameters are selected using validation PR-AUC only. **All combinations are selected before any held-out test metrics are computed.** Once selection is locked, the chosen model for each combination is scored once on each evaluation set.

In [ ]:
# 5.1 Select all hyperparameters on validation first; only then score held-out sets
MODELS, MODEL_SCORERS = {}, {}
val_se = EVAL["validation"]["se"]

def val_ap(score):
    return val_se.metrics(val_se.pack(score))["ap"]

def select_and_register(family, fit_fn, score_fn, grid, policy, fs):
    Xtr, ytr = DATA[(policy, fs)]["train"]
    Xva = DATA[(policy, fs)]["validation"]
    best = None
    for params in grid:
        t0 = time.time()
        model = fit_fn(params).fit(Xtr, ytr)
        # Selection log intentionally records validation only.
        SCORING_LOG.append((f"{family}|{fs}|{policy}|{params}", "validation"))
        ap = val_ap(score_fn(model, Xva))
        SEL_ROWS.append({"family": family, "feature_set": fs, "policy": policy, "params": str(params),
                         "val_ap": round(ap, 5), "seconds": round(time.time() - t0, 1)})
        if best is None or ap > best[0]:
            best = (ap, params.copy(), model)
    ap, params, model = best
    key = f"{family}|{fs}|{policy}"
    MODELS[key] = model
    MODEL_SCORERS[key] = score_fn
    INFO[key] = {"model": family, "feature_set": fs, "policy": policy, "params": params}
    print(f"  {key:28s} selected val AP={ap:.4f}  params={params}")

lr_fit = lambda p: LogisticRegression(C=p["C"], class_weight="balanced", max_iter=500, random_state=CONFIG["SEED"])
lr_score = lambda m, X: m.decision_function(X)
hgb_fit = lambda p: HistGradientBoostingClassifier(
    learning_rate=p["learning_rate"], max_depth=p["max_depth"], max_iter=p["max_iter"],
    l2_regularization=1.0, class_weight="balanced", early_stopping=False, random_state=CONFIG["SEED"])
hgb_score = lambda m, X: m.predict_proba(X)[:, 1]

for policy in CONFIG["POLICIES"]:
    for fs in CONFIG["FEATURE_SETS"]:
        print(f"[validation selection: {policy} / {fs}]")
        select_and_register("logreg", lr_fit, lr_score, [{"C": c} for c in CONFIG["LR_GRID_C"]], policy, fs)
        select_and_register("hgb", hgb_fit, hgb_score, CONFIG["HGB_GRID"], policy, fs)

SELECTION = pd.DataFrame(SEL_ROWS)
SELECTION.to_csv(P2_DIR / "results" / "model_selection_validation.csv", index=False)
record("M-0 all model/hyperparameter selection completed using validation only",
       len(SCORING_LOG) == len(SELECTION) and all(split == "validation" for _, split in SCORING_LOG),
       f"{len(SELECTION)} candidate fits; no held-out metric has been computed by the selection loop")

# Freeze all choices before computing a held-out metric.
for key, p in BASELINE_DEFS.items():
    if p["model"] == "prevalence":
        score_by_set = {s: np.zeros(len(EVAL[s]["y"]), dtype="float64") for s in CONFIG["EVAL_SETS"]}
    else:
        feat_idx = col[p["feature"]]
        score_by_set = {s: p["sign"] * RAW[s][:, feat_idx].astype("float64") for s in CONFIG["EVAL_SETS"]}
    SCORES[key] = score_by_set

for key, model in MODELS.items():
    fs, policy = INFO[key]["feature_set"], INFO[key]["policy"]
    scorer = MODEL_SCORERS[key]
    SCORES[key] = {s: scorer(model, DATA[(policy, fs)][s]) for s in CONFIG["EVAL_SETS"]}

print("\nAll candidate choices locked. Final selected models and pre-specified baselines scored on evaluation sets.")

## 6. Thresholds, point estimates, and paired machine-level bootstrap

Thresholds are fitted on validation scores only. The same bootstrap weight draws are shared among models within each evaluation set, enabling paired comparisons. Bootstrap units are machines to avoid treating thousands of hourly rows from the same machine as independent observations.

In [ ]:
# 6.1  Thresholds from validation, point estimates and paired cluster bootstrap for every model x eval set
B, LEVEL = CONFIG["BOOTSTRAP_B"], CONFIG["CI_LEVEL"]
W = {s: draw_weights(EVAL[s]["se"].M, B, CONFIG["SEED"] + 17 * i) for i, s in enumerate(CONFIG["EVAL_SETS"])}   # shared across models

THR, POINT, REPS = {}, {}, {}
t0 = time.time()
for key in SCORES:
    pk_val = val_se.pack(SCORES[key]["validation"])
    # Constant prevalence has no operational threshold; others choose F1 threshold on validation only.
    THR[key] = None if INFO[key]["model"] == "prevalence" else val_se.best_f1_threshold(pk_val)
    for s in CONFIG["EVAL_SETS"]:
        se = EVAL[s]["se"]
        pk = se.pack(SCORES[key][s])
        POINT[(key, s)] = se.metrics(pk, None, THR[key])
        reps = se.bootstrap(pk, THR[key], W[s])
        POINT[(key, s)]["lift"] = POINT[(key, s)]["ap"] / POINT[(key, s)]["prevalence"]
        reps["lift"] = reps["ap"] / reps["prevalence"]
        REPS[(key, s)] = reps
    print(f"  {key:30s} done ({time.time() - t0:.0f}s)")

rows = []
for (key, s), p in POINT.items():
    info = INFO[key]
    for mname in ["ap", "lift", "roc_auc", "precision", "recall", "f1", "fpr", "event_recall"]:
        metric_values = REPS[(key, s)][mname].dropna().to_numpy(dtype="float64")
        lo, hi = ci(metric_values, LEVEL) if len(metric_values) else (np.nan, np.nan)
        rows.append({"model_key": key, "model": info["model"], "feature_set": info["feature_set"], "policy": info["policy"],
                     "eval_set": s, "metric": mname, "value": p[mname], "ci_low": lo, "ci_high": hi,
                     "events": EVAL[s]["se"].n_events, "machines": EVAL[s]["se"].M,
                     # prep_v3's held-out calibration artifact is disjoint from all site-evaluation rows.
                     "calibration_overlap": False})
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(P2_DIR / "results" / "baseline_results_long.csv", index=False)
print(f"\nResults table: {len(RESULTS):,} rows saved. Bootstrap: B={B}, level={LEVEL:.0%}, resampling machines.")

## 7. Results

`test_site_full` is the primary held-out-site result because it has more distinct failure events than the nested late subsets. The stricter `test_site_post`, the temporal `test_time_late`, and `test_site_late` are reported separately as secondary/sensitivity sets. Do not treat nested test sets as independent replications.

In [ ]:
# 7.1 Ranked tables (primary metric: PR-AUC)
def fmt(v, lo, hi):
    if not np.isfinite(v) or not np.isfinite(lo) or not np.isfinite(hi):
        return "N/A"
    return f"{v:.3f} [{lo:.3f}, {hi:.3f}]"

def table(eval_set):
    d = RESULTS[RESULTS["eval_set"] == eval_set]
    piv = d.pivot_table(index="model_key", columns="metric", values=["value", "ci_low", "ci_high"])
    def get_value(model_key, stat, metric):
        col_key = (stat, metric)
        return piv.loc[model_key, col_key] if col_key in piv.columns else np.nan
    def get_vector(stat, metric):
        col_key = (stat, metric)
        if col_key not in piv.columns:
            return np.full(len(piv.index), np.nan, dtype="float64")
        return piv[col_key].to_numpy(dtype="float64")
    out = pd.DataFrame({
        "PR-AUC [95% CI]": [fmt(get_value(k, "value", "ap"), get_value(k, "ci_low", "ap"), get_value(k, "ci_high", "ap")) for k in piv.index],
        "lift": np.round(get_vector("value", "lift"), 2),
        "ROC-AUC": np.round(get_vector("value", "roc_auc"), 3),
        "event recall": np.round(get_vector("value", "event_recall"), 2),
        "FPR": np.round(get_vector("value", "fpr"), 3),
        "_ap": get_vector("value", "ap")}, index=piv.index)
    return out.sort_values("_ap", ascending=False).drop(columns="_ap")

for split in ["test_site_full", "test_site_post", "test_time_late", "test_site_late"]:
    e = EVAL[split]["se"]
    note = "  <-- FRAGILE (few distinct events)" if e.n_events < CONFIG["FRAGILE_EVENTS"] else ""
    print(f"\n=== {split}: {e.M} machines, {e.n_events} distinct failure events, prevalence {100 * EVAL[split]['y'].mean():.2f}%{note} ===")
    print(table(split).to_string())
print("\nValidation is used for model/threshold selection, not as a final generalization claim.")
print("Constant prevalence baseline: threshold-dependent columns are N/A by design.")

In [ ]:
# 7.2  Paired comparisons (same bootstrap resamples): maintenance ablation, normalization policy, model vs heuristics
def paired(a, b, s, metric="ap"):
    d_all = REPS[(a, s)][metric].to_numpy(dtype="float64") - REPS[(b, s)][metric].to_numpy(dtype="float64")
    d = d_all[np.isfinite(d_all)]
    lo, hi = ci(d, LEVEL) if len(d) else (np.nan, np.nan)
    return POINT[(a, s)][metric] - POINT[(b, s)][metric], lo, hi

MATERIAL = 0.005          # a PR-AUC difference below half a percentage point is treated as negligible in practice
def verdict(lo, hi, d):
    if not (lo > 0 or hi < 0):
        return "no clear difference"
    return "differs (CI excludes 0)" if abs(d) >= MATERIAL else "CI excludes 0 but negligible (<0.005)"

CMP_ROWS = []
def add(kind, a, b, s):
    d, lo, hi = paired(a, b, s)
    CMP_ROWS.append({"comparison": kind, "A": a, "B": b, "eval_set": s, "delta_AP(A-B)": round(d, 4),
                     "ci_low": round(lo, 4), "ci_high": round(hi, 4), "verdict": verdict(lo, hi, d)})

fair_sets = ["test_site_full", "test_site_post", "test_time_late", "test_site_late"]
for fam in ["logreg", "hgb"]:
    for pol in CONFIG["POLICIES"]:
        for s in fair_sets:
            add("maintenance ablation (full - no_maintenance)", f"{fam}|full|{pol}", f"{fam}|no_maintenance|{pol}", s)
    for fs in CONFIG["FEATURE_SETS"]:
        for s in fair_sets:
            add("normalization (global - local)", f"{fam}|{fs}|global", f"{fam}|{fs}|local", s)
# Choose the comparator using validation only; never pick a heuristic based on held-out test AP.
best_heur = max(["heur_error_recency|full|none", "heur_maint_recency|full|none"], key=lambda k: POINT[(k, "validation")]["ap"])
for fam in ["logreg", "hgb"]:
    for s in ["test_site_full", "test_site_post", "test_time_late", "test_site_late"]:
        add(f"model vs best heuristic ({best_heur.split('|')[0]})", f"{fam}|full|global", best_heur, s)
CMP = pd.DataFrame(CMP_ROWS)
CMP.to_csv(P2_DIR / "results" / "paired_comparisons.csv", index=False)
with pd.option_context("display.width", 250, "display.max_colwidth", 60):
    for kind in CMP["comparison"].unique():
        print(f"\n--- {kind} ---")
        print(CMP[CMP["comparison"] == kind].drop(columns="comparison").to_string(index=False))

print(f"\nNote: {len(CMP)} paired comparisons were made without multiplicity correction. Treat individual CI flags as exploratory; prioritize effect sizes and consistent patterns. test_site_post and test_site_late are subsets of test_site_full, so they are not independent replications.")

In [ ]:
# 8.1 Integrity checks, save artifacts, write report
record("M-1 model selection used the validation set only",
       all(split == "validation" for _, split in SCORING_LOG),
       f"{len(SCORING_LOG)} candidate fits scored on validation only")

ok_thr = all(
    THR[k] is None if INFO[k]["model"] == "prevalence" else
    abs(THR[k] - val_se.best_f1_threshold(val_se.pack(SCORES[k]["validation"]))) < 1e-12
    for k in SCORES
)
record("M-2 thresholds are validation-derived; prevalence has no threshold", ok_thr, f"{sum(v is not None for v in THR.values())} operational thresholds")

ok_prev = all(
    abs(POINT[("prevalence|full|none", s)]["ap"] - POINT[("prevalence|full|none", s)]["prevalence"]) < 1e-12
    and abs(POINT[("prevalence|full|none", s)]["roc_auc"] - 0.5) < 1e-12
    for s in CONFIG["EVAL_SETS"]
)
record("M-3 prevalence baseline: AP equals prevalence and ROC-AUC equals 0.5", ok_prev)

ap_rows = RESULTS[RESULTS["metric"] == "ap"]
inside = ap_rows.apply(lambda r: r["ci_low"] - 1e-9 <= r["value"] <= r["ci_high"] + 1e-9, axis=1)
record("M-4 AP point estimates are usually within machine-bootstrap CIs", bool(inside.mean() > 0.90),
       f"{inside.mean():.1%} of {len(inside)} AP intervals contain point estimate")

record("M-5 local held-out calibration does not overlap any site-evaluation row",
       not cal_overlap.shape[0] and TARGET not in calibration.columns,
       f"overlap rows={len(cal_overlap)}; test_site_full starts {site_full_raw['datetime'].min()}")

# Persist predictions only after choices were locked and the evaluation tables were computed.
for key in SCORES:
    parts = []
    for split in CONFIG["EVAL_SETS"]:
        m = EVAL[split]["meta"][["machineID", "datetime"]].copy()
        m["eval_set"] = split
        m["score"] = SCORES[key][split].astype("float32")
        parts.append(m)
    pd.concat(parts, ignore_index=True).to_parquet(P2_DIR / "predictions" / (key.replace("|", "__") + ".parquet"), index=False)
for key, mdl in MODELS.items():
    joblib.dump(mdl, P2_DIR / "models" / (key.replace("|", "__") + ".joblib"))

# Select the centralized comparator per feature set and normalization policy using validation AP only.
best_key = max((k for k in SCORES if INFO[k]["model"] in ("logreg", "hgb")),
               key=lambda k: POINT[(k, "validation")]["ap"])
REFERENCES = {}
for fs in CONFIG["FEATURE_SETS"]:
    for pol in CONFIG["POLICIES"]:
        candidates = [f"logreg|{fs}|{pol}", f"hgb|{fs}|{pol}"]
        REFERENCES[f"{fs}|{pol}"] = max(candidates, key=lambda k: POINT[(k, "validation")]["ap"])

primary = "test_site_full"
rep = {
    "phase": "Phase 2 centralized baselines",
    "input_prep_version": "prep_v3",
    "output_directory": str(P2_DIR),
    "config": CONFIG,
    "evaluation_protocol": {
        "selection_set": "validation",
        "primary_held_out_site_set": primary,
        "secondary_sets": ["test_site_post", "test_time_late", "test_site_late"],
        "nested_sets": {"test_site_post": "subset of test_site_full", "test_site_late": "subset of test_site_full"},
        "calibration_overlap_in_test_site_full": False,
        "bootstrap_unit": "machine",
        "constant_prevalence_threshold": None,
    },
    "eval_sets": EVAL_TABLE.to_dict("records"),
    "selected_hyperparameters": {k: INFO[k]["params"] for k in SCORES},
    "heuristic_definitions": {k: v for k, v in BASELINE_DEFS.items()},
    "thresholds_from_validation": THR,
    "best_model_by_validation_ap": best_key,
    "centralized_reference_per_feature_set_and_policy": REFERENCES,
    "headline_primary_set": {k: {"ap": POINT[(k, primary)]["ap"], "lift": POINT[(k, primary)]["lift"],
                                 "ci": ci(REPS[(k, primary)]["ap"].dropna().to_numpy(), LEVEL)} for k in SCORES},
    "checks": CHECKS,
    "limitations": [
        "Validation has a limited number of distinct failure events; hyperparameter and threshold choices can be noisy.",
        "Machine-cluster bootstrap intervals reflect variation across observed machines, not across independently sampled factory partitions.",
        "test_site_post and test_site_late are nested subsets of test_site_full, not independent replications.",
        "The held-out local scaler uses a separate unlabeled calibration period; local normalization is unsupervised adaptation, not a privacy guarantee.",
        "Maintenance recency may partly encode time since the last failure because replacement records often coincide with failures; report full versus no_maintenance.",
        "The primary split uses simulated factories from one dataset, not records from independently operated real factories.",
        "Centralized models are reference baselines, not federated learning results.",
        "No formal privacy mechanism, knowledge graph, or LLM explanation module is implemented in this phase.",
    ],
}
(P2_DIR / "reports" / "phase2_baselines_report.json").write_text(json.dumps(rep, indent=2, default=str))

# Human-readable report for reproducible project hand-off.
fmt_ci = lambda k, s: f"{POINT[(k, s)]['ap']:.3f} [{ci(REPS[(k, s)]['ap'].dropna().to_numpy(), LEVEL)[0]:.3f}, {ci(REPS[(k, s)]['ap'].dropna().to_numpy(), LEVEL)[1]:.3f}]"
md = [
    "# FedKG-Maint Phase 2 — Centralized Baselines (prep_v3)", "",
    f"- Primary held-out-site set: `{primary}`", "- Selection/threshold set: `validation` only",
    f"- Machine bootstrap replicates: {CONFIG['BOOTSTRAP_B']}", "",
    "## Centralized reference selected on validation",
    f"Best overall by validation AP: `{best_key}`; primary-set PR-AUC = {fmt_ci(best_key, primary)}", "",
    "| Feature set / policy | Validation-selected reference | Primary test PR-AUC [95% CI] |",
    "|---|---|---|",
]
for combo, key in REFERENCES.items():
    md.append(f"| `{combo}` | `{key}` | {fmt_ci(key, primary)} |")
md += ["", "## Primary held-out-site ranking", "", "| Model | PR-AUC [95% CI] | Lift over prevalence | Event recall |", "|---|---:|---:|---:|"]
primary_rows = RESULTS[(RESULTS["eval_set"] == primary) & (RESULTS["metric"] == "ap")].sort_values("value", ascending=False)
for _, row in primary_rows.iterrows():
    key = row["model_key"]
    lift = POINT[(key, primary)]["lift"]
    er = POINT[(key, primary)]["event_recall"]
    md.append(f"| `{key}` | {fmt_ci(key, primary)} | {lift:.2f} | {'N/A' if not np.isfinite(er) else f'{er:.3f}'} |")
md += ["", "## Interpretation notes", "",
        "`test_site_post` and `test_site_late` are nested within `test_site_full`; do not treat them as independent replications.",
        "The prevalence baseline has no decision threshold, so thresholded metrics are N/A.",
        "Maintenance-recency contribution must be assessed with the predeclared `full` vs `no_maintenance` comparisons.",
        "Centralized scores are reference values for later federated experiments; they are not federated-learning results.", ""]
(P2_DIR / "reports" / "phase2_baselines_report.md").write_text("\n".join(md))

pr = lambda k, s: fmt_ci(k, s)
prev = POINT[("prevalence|full|none", primary)]["prevalence"]
print("\n================ READOUT (primary held-out set: test_site_full) ================")
print(f"Prevalence baseline PR-AUC: {prev:.3f}")
for k in ["heur_error_recency|full|none", "heur_maint_recency|full|none", "logreg|full|global", "hgb|full|global",
          "logreg|no_maintenance|global", "hgb|no_maintenance|global"]:
    print(f"  {k:34s} PR-AUC {pr(k, primary)}")
print("\nCentralized references for Phase 3 (selected on validation only):")
for combo, k in REFERENCES.items():
    print(f"  {combo:22s} -> {k:30s} PR-AUC {pr(k, primary)}")
print(f"\nReports saved under: {P2_DIR / 'reports'}")
print("PHASE 2 BASELINES COMPLETE — all checks PASS.")

## Done — hand-off to Phase 3

Use the `centralized_reference_per_feature_set_and_policy` field in `reports/phase2_baselines_report.json` when comparing federated models. Each federated result must be compared with the centralized reference selected on validation for the same feature set and normalization policy.

This version deliberately removes post-hoc feature-group experiments that reported many extra scores on held-out test sets. The primary maintenance ablation remains the predeclared `full` versus `no_maintenance` comparison and is reported with paired machine-bootstrap intervals. Do not select new features, thresholds, or model settings from held-out results.

The main output folder is `phase2_baselines_prep_v3/`, leaving older artifacts untouched. The notebook must be run top-to-bottom on the same Google Drive project that contains the completed `prep_v3/` outputs; syntax validation alone does not establish final metric correctness.